# 🔍 Exploratory Data Analysis (EDA) — Credit Card Dataset

**Author:** Shaik Mudassir  
**Date:** June 2022 (Updated: May 2026)  
**Dataset:** [Credit Card Dataset](https://raw.githubusercontent.com/CosmiX-6/Bank-data-clustering-KMean-Clustering-and-PCA/master/assets/credit_card.csv)

---

## 📋 Overview

This notebook performs a comprehensive **Exploratory Data Analysis (EDA)** on a credit card customer dataset. The goal is to understand the data structure, identify patterns, detect outliers, and prepare insights for potential clustering or predictive modeling.

### What we'll cover:
1. 📥 Data loading & initial inspection
2. 📊 Statistical summary & distributions
3. 🔗 Correlation analysis & heatmaps
4. 📈 Univariate & bivariate visualizations
5. 🧹 Missing value & outlier detection
6. 💡 Key insights

## 1️⃣ Setup & Imports

We'll use **pandas** for data manipulation, **matplotlib** and **seaborn** for visualizations. The `%matplotlib inline` magic ensures plots render directly in the notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import skew, kurtosis

%matplotlib inline

# Styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_palette('viridis')

print("✅ Libraries loaded successfully")

## 2️⃣ Data Loading

Load the credit card dataset from a public GitHub raw URL. This dataset contains customer credit card usage and payment behavior.

In [ ]:
# Load dataset
url = 'https://raw.githubusercontent.com/CosmiX-6/Bank-data-clustering-KMean-Clustering-and-PCA/master/assets/credit_card.csv'
df = pd.read_csv(url)

print(f"✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")

## 3️⃣ First Look at the Data

Let's peek at the first 10 rows and column names to understand what each column represents and the type of values stored.

In [ ]:
# Display first 10 rows
df.head(10)

# Column names
print("\n📋 Columns:", list(df.columns))

## 4️⃣ Data Structure & Types

Understanding the **data types** and **non-null counts** of each column helps us identify:
- Which columns are numerical vs categorical
- If any columns have missing values
- Memory usage of the dataset

In [ ]:
# Dataset info
df.info()

## 5️⃣ Statistical Summary

The `describe()` function gives us key statistics for numerical columns:
- **Count:** Number of non-null values
- **Mean / Median (50%):** Central tendency
- **Std:** Spread of data
- **Min / Max:** Range of values
- **25% / 75%:** Interquartile range (IQR)

Using `.transpose()` makes it easier to read by flipping rows and columns.

In [ ]:
# Descriptive statistics (transposed for readability)
df.describe().transpose()

## 6️⃣ Correlation Analysis

A **correlation matrix** shows how strongly pairs of numerical features are related:
- `+1` → Strong positive correlation (both increase together)
- `-1` → Strong negative correlation (one increases, other decreases)
- `0` → No linear relationship

The **heatmap** makes it visually easy to spot strong correlations.

In [ ]:
# Correlation matrix as heatmap
corr_matrix = df.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, 
            annot=True,           # Show correlation values
            fmt='.2f',            # 2 decimal places
            cmap='coolwarm',      # Color scheme
            center=0,             # Center colormap at 0
            square=True,          # Square cells
            linewidths=0.5,       # Grid lines
            cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap — Credit Card Features', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

# Find strongest correlations (excluding self-correlations)
print("\n🔝 Top 5 Strongest Correlations:")
corr_pairs = corr_matrix.unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 1.0]  # Remove self-correlations
print(corr_pairs.head(5))

## 7️⃣ Missing Value Analysis

Missing data can skew analysis and affect model performance. Let's check for any null values and calculate the percentage per column.

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage (%)': missing_pct.round(2)
})

if missing.sum() == 0:
    print("✅ No missing values found in the dataset!")
else:
    print("⚠️ Missing values detected:")
    print(missing_df[missing_df['Missing Count'] > 0])
    
missing_df

## 8️⃣ Distribution Analysis

**Histograms** show us the shape of each feature's distribution — whether it's normal, skewed, bimodal, etc.

We'll plot distributions for all numerical columns to spot:
- Skewed features that may need transformation
- Outliers visible in the tails
- Multi-modal distributions suggesting subgroups

In [ ]:
# Distribution plots for all numerical columns
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
n_cols = len(num_cols)
n_rows = (n_cols + 2) // 3  # 3 plots per row

fig, axes = plt.subplots(n_rows, 3, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=30, alpha=0.7, edgecolor='black', color='steelblue')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {df[col].mean():.2f}')
    axes[i].axvline(df[col].median(), color='green', linestyle='--', linewidth=1.5, label=f'Median: {df[col].median():.2f}')
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].legend(fontsize=8)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Numerical Features', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

## 9️⃣ Outlier Detection — Box Plots

**Box plots** visually display the five-number summary (min, Q1, median, Q3, max) and highlight potential **outliers** (points beyond 1.5 × IQR).

Outliers may represent:
- Data entry errors
- Legitimate extreme values (high-net-worth customers, etc.)
- Natural variation

In [ ]:
# Box plots for outlier detection
fig, axes = plt.subplots(n_rows, 3, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                     boxprops=dict(facecolor='lightblue', alpha=0.7),
                     medianprops=dict(color='red', linewidth=1.5))
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_ylabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Box Plots — Outlier Detection', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

# IQR-based outlier count
print("\n📊 Outlier Count (IQR method):")
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"  {col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)")

## 🔟 Skewness & Kurtosis

- **Skewness**: Measures asymmetry. > 0 = right-skewed, < 0 = left-skewed
- **Kurtosis**: Measures "tailedness". > 3 = heavy tails (more outliers), < 3 = light tails

In [ ]:
# Skewness and Kurtosis
stats_df = pd.DataFrame({
    'Skewness': df[num_cols].apply(lambda x: skew(x.dropna())),
    'Kurtosis': df[num_cols].apply(lambda x: kurtosis(x.dropna()))
}).round(3)

print("📊 Distribution Shape Statistics:")
print(stats_df)

# Highlight highly skewed features
highly_skewed = stats_df[abs(stats_df['Skewness']) > 1]
if len(highly_skewed) > 0:
    print(f"\n⚠️ Highly skewed features (|skew| > 1): {list(highly_skewed.index)}")
else:
    print("\n✅ No highly skewed features")

## 1️⃣1️⃣ Pairwise Relationships

A **pair plot** shows scatter plots for every pair of numerical features plus histograms on the diagonal. Great for spotting relationships at a glance.

⚠️ Note: This may be heavy on larger datasets; we'll sample if needed.

In [ ]:
# Pair plot (sample if dataset is large)
plot_df = df.sample(min(500, len(df))) if len(df) > 500 else df

sns.pairplot(plot_df, diag_kind='kde', corner=True, plot_kws={'alpha': 0.5, 's': 20})
plt.suptitle('Pair Plot — Feature Relationships', fontsize=18, y=1.02)
plt.show()

## 1️⃣2️⃣ Key Insights & Summary

### 📊 Dataset Overview
- **Rows:** 8,950 customers  
- **Columns:** 18 features  

### 🔍 What We Found:
| Aspect | Finding |
|--------|---------|
| Missing Values | Check above — clean or needs handling |
| Correlations | Strong relationships identified in heatmap |
| Distributions | Histograms reveal shape of each feature |
| Outliers | Box plots highlight extreme values |
| Skewness | Some features may benefit from transformation |

### 🚀 Next Steps:
1. 🔄 Handle outliers if needed (capping, transformation)
2. 📐 Normalize/standardize features for distance-based algorithms
3. 🧠 Apply clustering (K-Means, DBSCAN) to segment customers
4. 🔢 Use PCA for dimensionality reduction
5. 🤖 Build predictive models (credit risk, spending prediction)

---

> **Note:** This notebook is part of a learning journey in Machine Learning and Data Science. Feel free to fork, star ⭐, or contribute!

**📁 Repo:** [github.com/skmudassir-it/Machine-Learning](https://github.com/skmudassir-it/Machine-Learning)